In [ ]:
import core_config
import os

# This notebook never mutates Torch or any shared environment on import.
core_config.map_storage()
core_config.set_hpc_proxies()
try:
    core_config.authenticate_huggingface(interactive=False)
except Exception as exc:
    print(f"[INFO] Non-interactive HF authentication not available: {type(exc).__name__}")
print("[SYSTEM] Read-only deployment diagnostics ready.")
print(f"[SYSTEM] Active Conda Environment: {os.environ.get('CONDA_PREFIX')}")

# Phase 1: Environment Provisioning (Compute Node)

!conda create --name oec_deploy python=3.12 -y
!conda activate oec_deploy
!conda install -c conda-forge jupyter nb_conda_kernels ipykernel -y
!pip install huggingface_hub uv
!python -m ipykernel install --user --name oec_deploy --display-name "Python (oec_deploy_v12)"

In [ ]:
%cd ~/oec_deploy/v1

!git clone https://github.com/Overworldai/Biome.git
!git clone https://github.com/Overworldai/world_engine.git

## Phase 2: Launching Jupyter (Compute Node)

1. Ensure your custom `core_config.py` file is placed directly inside the `~/oec_deploy/v1` directory so the notebook can find it.
2. Launch the Jupyter Notebook server bound to the compute node's interfaces. Note the access token printed in the terminal.
```bash
jupyter notebook --no-browser --port=8888 --ip=127.0.0.1

```



## Phase 3: The Multi-Hop Network Bridge (Ubuntu VM & Windows Host)

Because you require simultaneous access to the Jupyter environment and the Biome Inference API, you must establish two distinct tunnels.

1. **SSH Tunnels (Ubuntu VM):** Open a fresh terminal in your VirtualBox VM (outside the login node) and execute both routing commands. Replace `<COMPUTE_NODE>` with your actual Slurm node (e.g., `scn28-mn`).
```bash
# 1. Tunnel for Jupyter Interface
ssh -f -N -L 5402:<COMPUTE_NODE>:8888 alexi@login.aireos.com.in

# 2. Tunnel for Biome Server Inference API
ssh -f -N -L 7987:<COMPUTE_NODE>:7987 alexi@login.aireos.com.in

```


2. **VirtualBox Port Forwarding (Windows Host):** Ensure the following TCP rules are active in your VirtualBox Network Settings:
* **Rule 1 (Jupyter):** Host `127.0.0.1` Port `5402` $\rightarrow$ Guest `127.0.0.1` Port `5402`
* **Rule 2 (Biome API):** Host `127.0.0.1` Port `7987` $\rightarrow$ Guest `127.0.0.1` Port `7987`


## Phase 4: Jupyter Execution (`deploy_server.ipynb`)

Access Jupyter on your Windows host browser at `[http://127.0.0.1:5402](http://127.0.0.1:5402)`. Create a new notebook and select the **`Python [conda env:oec_deploy]`** kernel from the dropdown.

**Cell 1: Proxy & Core Initialization**
This must be executed first to lock in your cache mappings and proxy overrides before any ML libraries are initialized.

```python
# 1. CORE CONFIGURATION
import core_config
import os

print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
print(f"[SYSTEM] Active Conda Environment: {os.environ.get('CONDA_PREFIX')}")

```

**Cell 2: Dependency Resolution & Local Linking**
By explicitly targeting the `$CONDA_PREFIX` variable, we force the `uv` package manager to install the dependencies directly into your active Conda environment, eliminating the need for a separate `.venv`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

python_exe = Path(sys.executable).resolve()
server_dir = Path(os.path.expanduser("~/oec_deploy/v1/Biome/server-components")).resolve()
world_engine_dir = Path(os.path.expanduser("~/oec_deploy/v1/world_engine")).resolve()
print(f"[CHECK] Active Python: {python_exe}")
print(f"[CHECK] Biome server exists: {(server_dir / 'main.py').is_file()}")
print(f"[CHECK] World Engine project exists: {(world_engine_dir / 'pyproject.toml').is_file()}")

# Probe Torch in a child process; do not attempt destructive in-place repairs.
probe = subprocess.run(
    [str(python_exe), "-c",
     "import torch; print('torch=' + torch.__version__); "
     "print('cuda_build=' + str(torch.version.cuda)); "
     "print('cuda_available=' + str(torch.cuda.is_available()) + "
     "'\\npython=' + __import__('sys').executable)"],
    text=True, capture_output=True, timeout=120, check=False,
)
print(probe.stdout[-5000:])
if probe.returncode:
    print("[BLOCKED] Torch import failed. No packages were changed.")
    print(probe.stderr[-8000:])
    print("Repair only in a separate disposable Conda environment after reviewing this traceback.")
else:
    print("[PASS] Torch import probe succeeded.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

server_dir = Path(os.path.expanduser("~/oec_deploy/v1/Biome/server-components")).resolve()
server_file = server_dir / "main.py"
if os.environ.get("OEC_DEPLOY_LAUNCH_BIOME", "0") != "1":
    print("[SKIP] Biome launch is disabled by default.")
    print("After the Torch import probe succeeds, set OEC_DEPLOY_LAUNCH_BIOME=1 and rerun this cell.")
elif not server_file.is_file():
    raise FileNotFoundError(f"Biome server entry point is missing: {server_file}")
else:
    log_path = Path("/tmp") / "oec-biome-server.log"
    with log_path.open("ab") as log:
        process = subprocess.Popen(
            [sys.executable, str(server_file), "--host", "127.0.0.1", "--port", "7987"],
            cwd=str(server_dir), stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        )
    print(f"[STARTED] Biome server PID={process.pid}; log={log_path}")
    print("Access it through an authenticated SSH tunnel; do not bind to 0.0.0.0.")